In [3]:
from operator import matmul
from xml.etree import ElementInclude
import pandas as pd
import numpy as np
from mass_charge_dict import ELEMENTS2Z, Z2ELEMENTS,elements_dict
from scipy import linalg
from math import log10 , floor
import os
import shutil
from functions import *


In [15]:
input_path_coord = 'tests/NH3/coord.xyz'
input_path_hess = 'tests/NH3/hessian'
input_path_dipm = 'tests/NH3/xyz_dipm.csv'

coord,head = import_coord(input_path_coord)
hessian = import_hess(input_path_hess,coord)
dipm = import_dipm(input_path_dipm)

dipm = dipm.iloc[:,:-3]
############
########### Rotation of coordinates and hessian into intermediate position
# Calculating center of mass 
s = center_mass(coord) 
# Translation of coordinate system
vec_trans(coord,s)

#vec_trans(dipm,s)
# Calculating moment of inertia
I = inert_tensor(coord)

# Calculating eigenvalues and eigenvectors 
eig_val,eig_vec = linalg.eigh(I)

# Check if the coordinate system is right-handed --> important for chirality

eig_vec = check_eig_vec(eig_vec)

# Rotating eigenvectors, so that highest values are positive

eig_vec = eig_vec_rot(eig_vec)

# Rotation of the coordinates and atomic dipole moments
coord = coord_rot(coord,eig_vec.copy())

dipm = coord_rot(dipm,eig_vec.copy())

# Construction of the rotation matrix of the hessian and the rotation
P = rotM_hess(eig_vec.copy(),coord)
############

hessian = matmul(matmul(P,hessian),np.transpose(P))

hess_OH = hessian[0:3,3:6]
R_euler = get_R_euler(coord,dipm,0,1)

coord_rot(coord,R_euler)
coord_rot(dipm,R_euler)

rotM_Z = rot_Z(1/2*np.pi)

print(hess_OH)
print(coord)
for i in range(2):
    coord_rot(coord,rotM_Z)
    hess_OH = matmul(matmul(rotM_Z,hess_OH),np.transpose(rotM_Z))
    print(hess_OH)
    print(coord)


[[-2.93376176e-01  2.24572773e-01  6.32100000e-07]
 [ 1.82035980e-01 -2.03830093e-01 -2.79700000e-07]
 [ 7.06400000e-07 -4.28200000e-07  0.00000000e+00]]
  atoms             x             y         z
0     O -2.941782e-17  5.551115e-17  0.480429
1     H  2.941782e-17 -5.551115e-17 -0.480429
2     H -1.015736e-16 -9.275341e-01  0.731289
[[-2.03830093e-01 -1.82035980e-01  2.79700000e-07]
 [-2.24572773e-01 -2.93376176e-01  6.32100000e-07]
 [ 4.28200000e-07  7.06400000e-07  0.00000000e+00]]
  atoms             x             y         z
0     O -5.551115e-17 -2.941782e-17  0.480429
1     H  5.551115e-17  2.941782e-17 -0.480429
2     H  9.275341e-01 -1.583687e-16  0.731289
[[-2.93376176e-01  2.24572773e-01 -6.32100000e-07]
 [ 1.82035980e-01 -2.03830093e-01  2.79700000e-07]
 [-7.06400000e-07  4.28200000e-07  0.00000000e+00]]
  atoms             x             y         z
0     O  2.941782e-17 -5.551115e-17  0.480429
1     H -2.941782e-17  5.551115e-17 -0.480429
2     H  2.151638e-16  9.275341e

In [8]:
molecule = 'H2O'
input_path_coord = f'tests/coord_only/{molecule}/start_coord/coord.xyz'
input_path_hess = f'tests/coord_only/{molecule}/start_coord/hessian'
input_path_dipm = f'tests/coord_only/{molecule}/start_coord/xyz_dipm.csv'

coord,head = import_coord(input_path_coord)
hessian = import_hess(input_path_hess,coord)
dipm = import_dipm(input_path_dipm)

dipm = dipm.iloc[:,:-3]
############
########### Rotation of coordinates and hessian into intermediate position
# Calculating center of mass 
s = center_mass(coord) 
# Translation of coordinate system
vec_trans(coord,s)

#vec_trans(dipm,s)
# Calculating moment of inertia
I = inert_tensor(coord)

# Calculating eigenvalues and eigenvectors 
eig_val,eig_vec = linalg.eigh(I)

# Check if the coordinate system is right-handed --> important for chirality

eig_vec = check_eig_vec(eig_vec)

# Rotating eigenvectors, so that highest values are positive

eig_vec = eig_vec_rot(eig_vec)

# Rotation of the coordinates and atomic dipole moments
coord = coord_rot(coord,eig_vec.copy())

dipm = coord_rot(dipm,eig_vec.copy())

# Construction of the rotation matrix of the hessian and the rotation
P = rotM_hess(eig_vec.copy(),coord)
############

hessian = matmul(matmul(P,hessian),np.transpose(P))

hess_OH = hessian[0:3,3:6]
R_euler = get_R_euler(coord,dipm,0,1)

coord_rot(coord,R_euler)
coord_rot(dipm,R_euler)

rotM_Z = rot_Z(1/2*np.pi)

gamma_list = np.arange(0.5,2.04,0.1)

coord_dist = coord.copy()
for i in gamma_list:
    file_save_path = f'tests/coord_only/{molecule}//{molecule}_{round(i,3)}/'
    if os.path.exists(file_save_path):
        shutil.rmtree(file_save_path)

    os.mkdir(file_save_path)

    file_save_path = f'tests/coord_only/{molecule}/{molecule}_{round(i,3)}/init_coord/'
    if os.path.exists(file_save_path):
        shutil.rmtree(file_save_path)

    os.mkdir(file_save_path)

    gamma = i
    A = 0
    B = 1

    coord_dist.iloc[B,1:] = (1-gamma) * coord.iloc[A,1:] + gamma * coord.iloc[B,1:]

    file_save_path_c = file_save_path + f'coord.xyz'

    if os.path.exists(file_save_path):
        shutil.rmtree(file_save_path)

    os.mkdir(file_save_path)


    f = open(file_save_path + f'coord.xyz',"w")

    f.write(head[0])
    f.write(head[1])
    f.close()
    coord_dist.to_csv(file_save_path +'coord.xyz', mode ='a',sep = '\t',header = None , index = False)


